Используем selenium для парсинга сайта hirify

In [ ]:
!pip install selenium

In [138]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import logging
import sys
import pandas as pd

https://discourse.jupyter.org/t/jupyter-does-not-print-logging-information/11790

In [ ]:
BASE_URL = "https://hirify.me"
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(message)s")
logger = logging.getLogger('selenium_scrapper')
logger.setLevel(logging.DEBUG)
def prepare_driver_page(period: str = "?period=week") -> webdriver.Chrome:
    driver = webdriver.Chrome()
    logger.info("Driver created")
    driver.get(BASE_URL + period)
    time.sleep(2)
    logger.info("Page loaded")
    settings_button = driver.find_element(By.CLASS_NAME, "settings-button")
    settings_button.click()
    filter_up = driver.find_element(By.CLASS_NAME, "filters-row")
    filters = filter_up.find_elements(By.TAG_NAME, "input")
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
        if not f.is_selected():
            f.click()
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
    logger.info("Filters applied")
    return driver


driver = prepare_driver_page()

2026-04-07 03:12:08,178 Driver created
2026-04-07 03:12:15,060 Page loaded
2026-04-07 03:12:15,180 Filter  is selected: None


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=146.0.7680.178)
Stacktrace:
0   chromedriver                        0x0000000100d5399c cxxbridge1$str$ptr + 3160104
1   chromedriver                        0x0000000100d4ba28 cxxbridge1$str$ptr + 3127476
2   chromedriver                        0x000000010081f904 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75036
3   chromedriver                        0x00000001007f83e0 chromedriver + 164832
4   chromedriver                        0x0000000100890288 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 536224
5   chromedriver                        0x00000001008a6004 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 625692
6   chromedriver                        0x000000010085d1a0 _RNvCskE0kXQ3GIWk_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 327096
7   chromedriver                        

In [140]:
def parse_page(driver: webdriver.Chrome):
    cards = driver.find_elements(By.CSS_SELECTOR, "div.vacancy-card[data-vacancy-id]")
    logger.info(f"Found {len(cards)} cards")
    return cards

cards = parse_page(driver)

2026-04-07 02:50:57,775 Found 15 cards


Переписываем все это на 3 функции: 1 итератор по стринице 2 парсер 1 карточки 3 безопасные getter 

In [ ]:
def safe_find_element(parent, by, value):
    try:
        return parent.find_element(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements(parent, by, value):
    try:
        return parent.find_elements(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements_text(parent, by, value):
    r = safe_find_elements(parent, by, value)
    if r:
        if isinstance(r, list):
            return [elem.text for elem in r]
        else:
            return r.text
    return None

def parse_card(card):
    logger.debug(f"Parsing card with id {card.get_attribute('data-vacancy-id')}") # Используем get_attribute на случай если у элемента нет детей
    vacancy_link = safe_find_element(card, By.CLASS_NAME, "vacancy-card-link")
    vacancy_title = safe_find_element(card, By.CLASS_NAME, "title")
    vacancy_tags_div = safe_find_element(card, By.CLASS_NAME, "common-tags")
    vacancy_tags = []
    if vacancy_tags_div:
        vacancy_tags = vacancy_tags_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_tags = [t.text for t in vacancy_tags]
    vacancy_company = safe_find_element(card, By.CLASS_NAME, "company").text
    vacancy_time = safe_find_element(card, By.CLASS_NAME, "date")
    vacancy_skills_div = safe_find_element(card, By.CLASS_NAME, "vacancy-tags")
    vacancy_skills = []
    if vacancy_skills_div:
        vacancy_skills = vacancy_skills_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_skills = [s.text for s in vacancy_skills]
    vacansy_salary = safe_find_elements_text(card, By.CLASS_NAME, "salary")
    return {
        "title": vacancy_title.text,
        "link": vacancy_link.get_attribute("href"),
        "tags": vacancy_tags,
        "company": vacancy_company,
        "time": vacancy_time.text,
        "skills": vacancy_skills,
        "salary": vacansy_salary
    }

def parse_vacancies(cards: list) -> list[dict]:
    vacancies = []
    for card in cards:
        parse_card(card)
        vacancies.append(parse_card(card))
    return vacancies

vacancies = parse_vacancies(cards)
df = pd.DataFrame(vacancies)
df.head(20)

2026-04-07 02:29:58,068 Parsing card with id 440455
2026-04-07 02:29:58,162 Parsing card with id 440455
2026-04-07 02:29:58,256 Parsing card with id 440454
2026-04-07 02:29:58,351 Parsing card with id 440454
2026-04-07 02:29:58,449 Parsing card with id 440453
2026-04-07 02:29:58,529 Parsing card with id 440453
2026-04-07 02:29:58,618 Parsing card with id 440452
2026-04-07 02:29:58,705 Parsing card with id 440452
2026-04-07 02:29:58,796 Parsing card with id 440451
2026-04-07 02:29:58,896 Parsing card with id 440451
2026-04-07 02:29:59,005 Parsing card with id 440450
2026-04-07 02:29:59,110 Parsing card with id 440450
2026-04-07 02:29:59,211 Parsing card with id 440449
2026-04-07 02:29:59,298 Parsing card with id 440449
2026-04-07 02:29:59,401 Parsing card with id 440448
2026-04-07 02:29:59,533 Parsing card with id 440448
2026-04-07 02:29:59,656 Parsing card with id 440447
2026-04-07 02:29:59,750 Parsing card with id 440447
2026-04-07 02:29:59,841 Parsing card with id 440446
2026-04-07 0

,title,link,tags,company,time,skills,salary
0,Head Of Experience Design (AI),https://hirify.me/jobs/440455-head-experience-...,"[remote (Europe), Spain, fulltime, head]",Company hidden,29 seconds ago,"[ai, ux, ui, product design, design system, +1...",None
1,Global Hr Operations Lead,https://hirify.me/jobs/440454-global-hr-operat...,"[remote (Europe)/hybrid, Armenia, fulltime, lead]",Company hidden,37 seconds ago,"[excel, automation, labor law, international h...",None
2,Senior Product Marketing Manager (SaaS),https://hirify.me/jobs/440453-senior-product-m...,"[remote (Europe), Spain, fulltime, senior]",Company hidden,37 seconds ago,"[saas, product marketing, stakeholder manageme...",None
3,Senior Backend Engineer (Platform),https://hirify.me/jobs/440452-senior-backend-e...,"[remote (Europe), Spain, fulltime, senior]",Company hidden,45 seconds ago,"[ai, java, kotlin, javascript, typescript, +6 ...",None
4,Talent Acquisition (Gamedev),https://hirify.me/jobs/440451-talent-acquisiti...,"[remote (USA), US, fulltime, middle]",Company hidden,51 seconds ago,"[recruiting, jira, confluence, talent acquisit...",None
5,Vp Marketing (SaaS),https://hirify.me/jobs/440450-vp-marketing-saas,"[remote (Global), Germany, fulltime, director]",Company hidden,1 minute ago,"[ai, seo, saas, product marketing, brand posit...",None
6,Senior Backend Engineer (Fintech),https://hirify.me/jobs/440449-senior-backend-e...,"[remote (Europe), Spain, fulltime, senior]",Company hidden,1 minute ago,"[ai, java, kotlin, kafka, linux, +4 skills]",None
7,Paid Acquisition Manager,https://hirify.me/jobs/440448-paid-acquisition...,"[remote (Europe), Spain, fulltime, middle]",Company hidden,1 minute ago,"[ai, excel, english, performance marketing, go...",None
8,Senior Machine Learning Engineer (AI),https://hirify.me/jobs/440447-senior-machine-l...,"[Poland, fulltime, senior]",Company hidden,2 minutes ago,"[ai, python, kubernetes, redis, kafka, +5 skills]",None
9,People Operations AI & Analytics Partner,https://hirify.me/jobs/440446-people-operation...,"[remote (Global), Germany, fulltime]",Company hidden,2 minutes ago,"[ai, data analysis, automation, people operati...",None


Итак погинация

In [151]:
next_page_button = safe_find_element(driver, By.CSS_SELECTOR, 'button[aria-label="Next Page"]')
#next_page_button.click()
next_page_button.get_attribute("aria-label")

'Next Page'

листаем до последней страницы и находим по devtools что появляется флаг disabled

In [146]:
next_page_button.get_attribute("disabled")

In [ ]:
def safe_find_element(parent, by, value):
    try:
        return parent.find_element(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements(parent, by, value):
    try:
        return parent.find_elements(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements_text(parent, by, value):
    r = safe_find_elements(parent, by, value)
    if r:
        if isinstance(r, list):
            return [elem.text for elem in r]
        else:
            return r.text
    return None

def parse_card(card):
    logger.debug(f"Parsing card with id {card.get_attribute('data-vacancy-id')}") # Используем get_attribute на случай если у элемента нет детей
    vacancy_link = safe_find_element(card, By.CLASS_NAME, "vacancy-card-link")
    vacancy_title = safe_find_elements_text(card, By.CLASS_NAME, "title")
    vacancy_tags_div = safe_find_element(card, By.CLASS_NAME, "common-tags")
    vacancy_tags = []
    if vacancy_tags_div:
        vacancy_tags = vacancy_tags_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_tags = [t.text for t in vacancy_tags]
    vacancy_company = safe_find_elements_text(card, By.CLASS_NAME, "company")
    vacancy_time = safe_find_element(card, By.CLASS_NAME, "date")
    vacancy_skills_div = safe_find_element(card, By.CLASS_NAME, "vacancy-tags")
    vacancy_skills = []
    if vacancy_skills_div:
        vacancy_skills = vacancy_skills_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_skills = [s.text for s in vacancy_skills]
    vacansy_salary = safe_find_elements_text(card, By.CLASS_NAME, "salary")
    return {
        "title": vacancy_title,
        "link": vacancy_link.get_attribute("href"),
        "tags": vacancy_tags,
        "company": vacancy_company,
        "time": vacancy_time.text,
        "skills": vacancy_skills,
        "salary": vacansy_salary
    }

def parse_vacancies(cards: list) -> list[dict]:
    vacancies = []
    for card in cards:
        parse_card(card)
        vacancies.append(parse_card(card))
    return vacancies

def parse_page(driver: webdriver.Chrome):
    cards = driver.find_elements(By.CSS_SELECTOR, "div.vacancy-card[data-vacancy-id]")
    logger.info(f"Found {len(cards)} cards")
    return cards

def go_to_next_page(driver):
    next_page_button = safe_find_element(driver, By.CSS_SELECTOR, 'button[aria-label="Next Page"]')
    if next_page_button and not next_page_button.get_attribute("disabled"):
        next_page_button.click()
        return True
    return False

In [ ]:
def parse_pages(driver):
    all_vacancies = []
    for i in range(1000):
        time.sleep(1)
        logger.info(f"Parsing page {i+1}")
        cards = parse_page(driver)
        vacancies = parse_vacancies(cards)
        all_vacancies.extend(vacancies)
        if not go_to_next_page(driver):
            break 
    return all_vacancies

logger.setLevel(logging.INFO)
driver = prepare_driver_page()
vacancies = parse_pages(driver)
df = pd.DataFrame(vacancies)
df.head(20)
df.to_csv("vacancies.csv", index=False)

2026-04-07 03:06:10,473 Driver created
2026-04-07 03:06:17,496 Page loaded
2026-04-07 03:06:17,699 Filters applied
2026-04-07 03:06:18,703 Parsing page 1
2026-04-07 03:06:18,710 Found 15 cards
2026-04-07 03:06:22,532 Parsing page 2
2026-04-07 03:06:22,539 Found 15 cards
2026-04-07 03:06:26,645 Parsing page 3
2026-04-07 03:06:26,651 Found 15 cards
2026-04-07 03:06:30,645 Parsing page 4
2026-04-07 03:06:30,650 Found 15 cards
2026-04-07 03:06:34,324 Parsing page 5
2026-04-07 03:06:34,333 Found 15 cards
2026-04-07 03:06:37,909 Parsing page 6
2026-04-07 03:06:37,916 Found 15 cards
2026-04-07 03:06:41,577 Parsing page 7
2026-04-07 03:06:41,582 Found 15 cards
2026-04-07 03:06:45,199 Parsing page 8
2026-04-07 03:06:45,207 Found 15 cards
2026-04-07 03:06:48,769 Parsing page 9
2026-04-07 03:06:48,777 Found 15 cards
2026-04-07 03:06:52,286 Parsing page 10
2026-04-07 03:06:52,294 Found 15 cards
2026-04-07 03:06:55,917 Parsing page 11
2026-04-07 03:06:55,925 Found 15 cards
2026-04-07 03:06:59,500 P

AttributeError: 'NoneType' object has no attribute 'text'

In [170]:
len(df)

15